# 19. CLIP VQA — Zero-Shot и Fine-Tuning (CLIP ViT-B/32 + SigLIP 2)

Четыре эксперимента на AI2D MCQ VQA (question-level, test = 3088 вопросов):

| # | Модель | Метод |
|---|--------|-------|
| A | CLIP ViT-B/32 | **Zero-shot** |
| B | CLIP ViT-B/32 | **Fine-tuning** на AI2D train |
| C | SigLIP 2 Base | **Zero-shot** |
| D | SigLIP 2 Base | **Fine-tuning** на AI2D train |

**Метод MCQ:** кодируем изображение + 4 варианта ответа → argmax cosine similarity.

**Режим (ячейка 1):** `RETRAIN=True` (по умолчанию) — **дообучение** CLIP и SigLIP на AI2D
train с прогресс-барами `tqdm` (текстовые, без ipywidgets — надёжно в VSCode); лучший по val
чекпоинт сохраняется и оценивается на test. `RETRAIN=False` — пропустить обучение и оценить
готовые чекпоинты. Все 4 метрики → `runs/clip_vqa_baseline/metrics_all.json`.

> Обучение на GPU (CLIP ~мин/эпоха, SigLIP заметно дольше). Кернел — **Python (data-cu124)**.

In [ ]:
# ── 0. Пути ───────────────────────────────────────────────────────────────────
from pathlib import Path
from PIL import Image
from tqdm import tqdm          # обычный tqdm: текстовые бары, без ipywidgets (надёжно в VSCode)
p = Path.cwd()
while p != p.parent:
    if (p / 'src' / 'vqa_retrieval').exists() and (p / 'notebooks').exists():
        break
    p = p.parent
ROOT     = p
EXTERNAL = ROOT.parent
print('ROOT:    ', ROOT)
print('EXTERNAL:', EXTERNAL)

In [24]:
# ── 1. Конфиг ─────────────────────────────────────────────────────────────────
MANIFEST     = EXTERNAL / 'ai2d' / 'prepared_v2' / 'manifest_hybrid.jsonl'
OUTPUT_DIR   = ROOT / 'runs' / 'clip_vqa_baseline'
CKPT_PATH    = OUTPUT_DIR / 'clip_finetuned.pt'
CKPT_SIG_PATH = OUTPUT_DIR / 'siglip2_finetuned.pt'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# RETRAIN=True  → дообучить CLIP и SigLIP на AI2D train (прогресс через tqdm.notebook),
#                 лучший по val чекпоинт сохраняется и затем оценивается на test.
# RETRAIN=False → пропустить обучение и оценить уже сохранённые чекпоинты.
RETRAIN = True

FT_EPOCHS     = 5
FT_LR         = 2e-6
FT_BATCH      = 32
FT_MAX_STEPS  = None   # None = все батчи эпохи
SIG_BATCH     = 16
SIG_LR        = 2e-6

# Для быстрой проверки цепочки: MAX_QUICK = 300 (иначе None = весь test 3088)
MAX_QUICK = None

print(f'RETRAIN={RETRAIN} | CLIP ft: {FT_EPOCHS}ep lr={FT_LR} bs={FT_BATCH} | SigLIP ft: bs={SIG_BATCH}')

RETRAIN=True | CLIP ft: 5ep lr=2e-06 bs=32 | SigLIP ft: bs=16


In [25]:
# ── 2. Загрузка данных ────────────────────────────────────────────────────────
import json
from pathlib import Path

N_OPTIONS = 4  # AI2D всегда 4-choice MCQ

def load_split(manifest, split, max_samples=None):
    samples = []
    for line in manifest.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        r = json.loads(line)
        if r.get('split') != split:
            continue
        opts = r.get('options', [])
        idx  = r.get('correct_option_idx')
        raw  = r['image_path']
        img  = Path(raw) if Path(raw).is_absolute() else EXTERNAL / raw
        if len(opts) != N_OPTIONS or idx is None or idx >= len(opts) or not img.exists():
            continue
        samples.append({
            'image_path': str(img),
            'question':   r['question'],
            'options':    opts,
            'label_idx':  int(idx),
        })
        if max_samples and len(samples) >= max_samples:
            break
    return samples

train_samples = load_split(MANIFEST, 'train')
val_samples   = load_split(MANIFEST, 'val')
test_samples  = load_split(MANIFEST, 'test', max_samples=MAX_QUICK)

print(f'Train: {len(train_samples)}, Val: {len(val_samples)}, Test: {len(test_samples)}')

Train: 11143, Val: 1268, Test: 3088


In [7]:
# ── 3. Загрузка CLIP ──────────────────────────────────────────────────────────
import torch
import clip

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

model_clip, preprocess_clip = clip.load('ViT-B/32', device=DEVICE)
model_clip = model_clip.float()
model_clip.eval()
print('CLIP ViT-B/32 loaded (FP32)')

Device: cuda
CLIP ViT-B/32 loaded (FP32)


---
## Часть A: CLIP ViT-B/32 — Zero-Shot (до обучения)

In [5]:


def eval_clip_vqa(model, preprocess, samples, device, desc='CLIP VQA'):
    model.eval()
    correct   = 0
    img_cache = {}
    with torch.no_grad():
        for item in tqdm(samples, desc=desc):
            ip = item['image_path']
            if ip not in img_cache:
                pil    = Image.open(ip).convert('RGB')
                tensor = preprocess(pil).unsqueeze(0).to(device)
                feat   = model.encode_image(tensor)
                img_cache[ip] = feat / feat.norm(dim=-1, keepdim=True)
            img_feat = img_cache[ip]

            texts     = [f'A diagram answer: {opt}' for opt in item['options']]
            tok       = clip.tokenize(texts, truncate=True).to(device)
            txt_feats = model.encode_text(tok)
            txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)

            pred = (img_feat @ txt_feats.T).squeeze(0).argmax().item()
            if pred == item['label_idx']:
                correct += 1
    return correct, len(samples), correct / len(samples)

c, t, acc_zeroshot = eval_clip_vqa(model_clip, preprocess_clip, test_samples, DEVICE,
                                    desc='Zero-Shot CLIP')
print(f'\nA. CLIP ViT-B/32 zero-shot: {acc_zeroshot:.4f}  ({c}/{t})')

Zero-Shot CLIP:   0%|          | 0/3088 [00:00<?, ?it/s]


A. CLIP ViT-B/32 zero-shot: 0.2992  (924/3088)


---
## Часть B: CLIP ViT-B/32 — Fine-Tuning на AI2D

In [13]:
# Dataset для fine-tuning
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class AI2DCLIPDataset(Dataset):
    def __init__(self, samples, preprocess):
        self.samples   = samples
        self.preprocess = preprocess

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item  = self.samples[idx]
        image = self.preprocess(Image.open(item['image_path']).convert('RGB'))
        texts = [f'A diagram answer: {opt}' for opt in item['options']]
        label = item['label_idx']
        return image, texts, label

def collate_fn(batch):
    images, texts_batch, labels = zip(*batch)
    images = torch.stack(images)
    flat_texts = [t for texts in texts_batch for t in texts]
    labels = torch.tensor(labels, dtype=torch.long)
    return images, flat_texts, labels

train_ds = AI2DCLIPDataset(train_samples, preprocess_clip)
val_ds   = AI2DCLIPDataset(val_samples, preprocess_clip)
train_loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True,
                          num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=FT_BATCH, shuffle=False,
                          num_workers=0, collate_fn=collate_fn)

print(f'Train batches: {len(train_loader)}, Val batches: {len(val_loader)}')

Train batches: 349, Val batches: 40


In [16]:
# Fine-tuning loop CLIP (FP32) — прогресс через tqdm.notebook
import copy, time
from tqdm.notebook import tqdm

def clip_forward(model, images, flat_texts, B, device):
    """Returns sims (B x N_OPTIONS) for a batch."""
    tok       = clip.tokenize(flat_texts, truncate=True).to(device)
    img_feats = model.encode_image(images)
    txt_feats = model.encode_text(tok)
    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
    txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)
    txt_feats = txt_feats.view(B, N_OPTIONS, -1).contiguous()
    return torch.bmm(img_feats.unsqueeze(1), txt_feats.transpose(1, 2)).squeeze(1)

if not RETRAIN:
    assert CKPT_PATH.exists(), f'Нет чекпоинта {CKPT_PATH}. Поставьте RETRAIN=True.'
    print(f'RETRAIN=False → обучение CLIP пропущено, используется {CKPT_PATH.name}')
else:
    optimizer    = torch.optim.AdamW(model_clip.parameters(), lr=FT_LR, weight_decay=0.01)
    best_val_acc = 0.0
    print(f'Fine-tuning CLIP ViT-B/32: {FT_EPOCHS} epochs, lr={FT_LR}, bs={FT_BATCH}\n')
    for epoch in range(1, FT_EPOCHS + 1):
        model_clip.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f'CLIP epoch {epoch}/{FT_EPOCHS}', leave=False)
        for step, (images, flat_texts, labels) in enumerate(pbar):
            images = images.to(DEVICE); labels = labels.to(DEVICE); B = images.size(0)
            optimizer.zero_grad()
            sims        = clip_forward(model_clip, images, flat_texts, B, DEVICE)
            logit_scale = model_clip.logit_scale.exp().clamp(max=100)
            loss        = F.cross_entropy(sims * logit_scale, labels)
            loss.backward(); optimizer.step(); total_loss += loss.item()
            pbar.set_postfix(loss=f'{total_loss/(step+1):.4f}')
            if FT_MAX_STEPS and step + 1 >= FT_MAX_STEPS:
                break
        model_clip.eval(); val_correct = 0
        with torch.no_grad():
            for images, flat_texts, labels in tqdm(val_loader, desc=f'  val {epoch}', leave=False):
                images = images.to(DEVICE); labels = labels.to(DEVICE); B = images.size(0)
                sims = clip_forward(model_clip, images, flat_texts, B, DEVICE)
                val_correct += (sims.argmax(dim=1) == labels).sum().item()
        val_acc = val_correct / len(val_samples); avg_loss = total_loss / (step + 1)
        mark = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(copy.deepcopy(model_clip.state_dict()), CKPT_PATH)
            mark = '  → saved best'
        print(f'Epoch {epoch}/{FT_EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}{mark}')
    print(f'\nBest CLIP val_acc: {best_val_acc:.4f}')

Fine-tuning CLIP ViT-B/32: 5 epochs, lr=2e-06, bs=32



CLIP epoch 1/5:   0%|          | 0/349 [00:00<?, ?it/s]

  val 1:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch 1/5  loss=1.0632  val_acc=0.4101  → saved best


CLIP epoch 2/5:   0%|          | 0/349 [00:00<?, ?it/s]

...

  val 4:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch 4/5  loss=0.6843  val_acc=0.3872


CLIP epoch 5/5:   0%|          | 0/349 [00:00<?, ?it/s]

  val 5:   0%|          | 0/40 [00:00<?, ?it/s]

Epoch 5/5  loss=0.5995  val_acc=0.3778

Best CLIP val_acc: 0.4101


In [ ]:
# Fine-tuning loop CLIP (FP32) — прогресс через обычный tqdm (без ipywidgets)
import copy, time
from tqdm import tqdm

def clip_forward(model, images, flat_texts, B, device):
    """Returns sims (B x N_OPTIONS) for a batch."""
    tok       = clip.tokenize(flat_texts, truncate=True).to(device)
    img_feats = model.encode_image(images)
    txt_feats = model.encode_text(tok)
    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
    txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)
    txt_feats = txt_feats.view(B, N_OPTIONS, -1).contiguous()
    return torch.bmm(img_feats.unsqueeze(1), txt_feats.transpose(1, 2)).squeeze(1)

if not RETRAIN:
    assert CKPT_PATH.exists(), f'Нет чекпоинта {CKPT_PATH}. Поставьте RETRAIN=True.'
    print(f'RETRAIN=False → обучение CLIP пропущено, используется {CKPT_PATH.name}')
else:
    optimizer    = torch.optim.AdamW(model_clip.parameters(), lr=FT_LR, weight_decay=0.01)
    best_val_acc = 0.0
    print(f'Fine-tuning CLIP ViT-B/32: {FT_EPOCHS} epochs, lr={FT_LR}, bs={FT_BATCH}\n')
    for epoch in range(1, FT_EPOCHS + 1):
        model_clip.train()
        total_loss = 0.0
        pbar = tqdm(train_loader, desc=f'CLIP epoch {epoch}/{FT_EPOCHS}', leave=False)
        for step, (images, flat_texts, labels) in enumerate(pbar):
            images = images.to(DEVICE); labels = labels.to(DEVICE); B = images.size(0)
            optimizer.zero_grad()
            sims        = clip_forward(model_clip, images, flat_texts, B, DEVICE)
            logit_scale = model_clip.logit_scale.exp().clamp(max=100)
            loss        = F.cross_entropy(sims * logit_scale, labels)
            loss.backward(); optimizer.step(); total_loss += loss.item()
            pbar.set_postfix(loss=f'{total_loss/(step+1):.4f}')
            if FT_MAX_STEPS and step + 1 >= FT_MAX_STEPS:
                break
        model_clip.eval(); val_correct = 0
        with torch.no_grad():
            for images, flat_texts, labels in tqdm(val_loader, desc=f'  val {epoch}', leave=False):
                images = images.to(DEVICE); labels = labels.to(DEVICE); B = images.size(0)
                sims = clip_forward(model_clip, images, flat_texts, B, DEVICE)
                val_correct += (sims.argmax(dim=1) == labels).sum().item()
        val_acc = val_correct / len(val_samples); avg_loss = total_loss / (step + 1)
        mark = ''
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(copy.deepcopy(model_clip.state_dict()), CKPT_PATH)
            mark = '  → saved best'
        print(f'Epoch {epoch}/{FT_EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}{mark}')
    print(f'\nBest CLIP val_acc: {best_val_acc:.4f}')

---
## Часть C: SigLIP 2 — Zero-Shot (современный CLIP, Google 2025)

SigLIP 2 использует **sigmoid loss** вместо softmax + обучен на 4B пар (multilingual).  
ImageNet zero-shot accuracy: **85%** против **63%** у CLIP ViT-B/32.  

Установка: `pip install transformers>=4.49`

In [29]:
# transformers 4.49.0 + sentencepiece уже установлены. НЕ апгрейдим (бережём пин pylate).
# Отключаем зондирование TF/Flax-бэкендов ДО первого import transformers — иначе на Windows
# первый импорт может надолго подвисать.
import os
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

import transformers, sentencepiece
print('transformers', transformers.__version__, '| sentencepiece', sentencepiece.__version__)

transformers 4.49.0 | sentencepiece 0.2.1


In [30]:
# Загрузка SigLIP 2
# В transformers 4.49.0 и AutoProcessor, и SiglipProcessor жёстко требуют SiglipTokenizer,
# но siglip2 использует Gemma-токенизатор → стандартный путь падает (vocab_file=None / TypeError).
# Грузим токенизатор и image-processor по отдельности и оборачиваем в совместимый процессор,
# отдающий BatchFeature с теми же ключами, что ждут ячейки C/D — ниже ничего менять не нужно.
# ВАЖНО: SigLIP обучался БЕЗ attention_mask (паддинг до фикс. длины, все токены) — поэтому
# маску НЕ возвращаем (return_attention_mask=False), как и штатный SiglipProcessor.
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor, BatchFeature

SIGLIP_MODEL = 'google/siglip2-base-patch16-224'
SIG_MAXLEN   = 64        # длина текста siglip
print(f'Loading {SIGLIP_MODEL}...')

_tok  = AutoTokenizer.from_pretrained(SIGLIP_MODEL)
_imgp = AutoImageProcessor.from_pretrained(SIGLIP_MODEL, use_fast=False)

class SiglipProcessorCompat:
    """Замена SiglipProcessor: image_processor (siglip) + Gemma-tokenizer."""
    def __init__(self, image_processor, tokenizer):
        self.image_processor = image_processor
        self.tokenizer = tokenizer
    def __call__(self, text=None, images=None, return_tensors='pt',
                 padding='max_length', truncation=True, max_length=SIG_MAXLEN, **kw):
        data = {}
        if images is not None:
            data.update(self.image_processor(images=images, return_tensors=return_tensors))
        if text is not None:
            data.update(self.tokenizer(text, return_tensors=return_tensors, padding=padding,
                                       truncation=truncation, max_length=max_length,
                                       return_attention_mask=False))
        return BatchFeature(data)

siglip_processor = SiglipProcessorCompat(_imgp, _tok)
siglip_model = AutoModel.from_pretrained(SIGLIP_MODEL).to(DEVICE)
siglip_model.eval()
print('SigLIP 2 loaded | tokenizer:', type(_tok).__name__, '| image:', type(_imgp).__name__)

Loading google/siglip2-base-patch16-224...
SigLIP 2 loaded | tokenizer: GemmaTokenizerFast | image: SiglipImageProcessor


In [31]:
# Оценка SigLIP 2 на MCQ VQA
correct_sig = 0
img_cache_sig = {}

with torch.no_grad():
    for item in tqdm(test_samples, desc='SigLIP 2 VQA'):
        # Формируем 4 (image, text) пары
        texts = [f'A diagram answer: {opt}' for opt in item['options']]
        pil   = Image.open(item['image_path']).convert('RGB')

        # SigLIP принимает список изображений и список текстов попарно
        # Нам нужна матрица: 1 изображение × 4 текста
        inputs = siglip_processor(
            text=texts,
            images=[pil] * 4,   # одно изображение, повторённое 4 раза
            return_tensors='pt',
            padding='max_length',
            truncation=True,
        ).to(DEVICE)

        outputs = siglip_model(**inputs)
        # logits_per_image: (4, 4) — берём диагональ (каждый image-text score)
        # Т.к. мы повторили одно изображение — все строки одинаковы
        logits = outputs.logits_per_image[0]
        pred   = logits.argmax().item()

        if pred == item['label_idx']:
            correct_sig += 1

acc_siglip = correct_sig / len(test_samples)
print(f'\nC. SigLIP 2 zero-shot: {acc_siglip:.4f}  ({correct_sig}/{len(test_samples)})')

SigLIP 2 VQA:   0%|          | 0/3088 [00:00<?, ?it/s]


C. SigLIP 2 zero-shot: 0.2843  (878/3088)


---
## Часть D: SigLIP 2 — Fine-Tuning на AI2D

In [33]:
# Dataset для SigLIP (processor вместо torchvision-трансформов)
from torch.utils.data import Dataset, DataLoader

class AI2DSigLIPDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item  = self.samples[idx]
        image = Image.open(item['image_path']).convert('RGB')
        texts = [f'A diagram answer: {opt}' for opt in item['options']]
        return image, texts, item['label_idx']

def siglip_collate_fn(batch):
    images_pil, texts_batch, labels = zip(*batch)
    flat_texts = [t for texts in texts_batch for t in texts]
    return list(images_pil), flat_texts, torch.tensor(labels, dtype=torch.long)

sig_train_ds     = AI2DSigLIPDataset(train_samples)
sig_val_ds       = AI2DSigLIPDataset(val_samples)
sig_train_loader = DataLoader(sig_train_ds, batch_size=SIG_BATCH, shuffle=True,
                               num_workers=0, collate_fn=siglip_collate_fn)
sig_val_loader   = DataLoader(sig_val_ds,   batch_size=SIG_BATCH, shuffle=False,
                               num_workers=0, collate_fn=siglip_collate_fn)

print(f'SigLIP Train batches: {len(sig_train_loader)}, Val batches: {len(sig_val_loader)}')

SigLIP Train batches: 697, Val batches: 80


In [34]:
# SigLIP 2 fine-tuning (FP32) — прогресс через tqdm.notebook
import copy, time
import torch.nn.functional as F
from tqdm.notebook import tqdm

def siglip_forward(model, processor, images_pil, flat_texts, B, device):
    images_rep = [img for img in images_pil for _ in range(N_OPTIONS)]
    inputs = processor(text=flat_texts, images=images_rep,
                       return_tensors='pt', padding='max_length', truncation=True).to(device)
    text_inputs = {k: v for k, v in inputs.items() if k != 'pixel_values'}
    img_feats = model.get_image_features(pixel_values=inputs['pixel_values'])
    txt_feats = model.get_text_features(**text_inputs)
    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
    txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)
    img_feats = img_feats.view(B, N_OPTIONS, -1)[:, 0, :].contiguous()
    txt_feats = txt_feats.view(B, N_OPTIONS, -1).contiguous()
    return torch.bmm(img_feats.unsqueeze(1), txt_feats.transpose(1, 2)).squeeze(1)

if not RETRAIN:
    assert CKPT_SIG_PATH.exists(), f'Нет чекпоинта {CKPT_SIG_PATH}. Поставьте RETRAIN=True.'
    print(f'RETRAIN=False → обучение SigLIP пропущено, используется {CKPT_SIG_PATH.name}')
else:
    sig_optimizer = torch.optim.AdamW(siglip_model.parameters(), lr=SIG_LR, weight_decay=0.01)
    sig_best_val  = 0.0
    print(f'Fine-tuning SigLIP 2: {FT_EPOCHS} epochs, lr={SIG_LR}, bs={SIG_BATCH}\n')
    for epoch in range(1, FT_EPOCHS + 1):
        siglip_model.train()
        total_loss = 0.0
        pbar = tqdm(sig_train_loader, desc=f'SigLIP epoch {epoch}/{FT_EPOCHS}', leave=False)
        for step, (images_pil, flat_texts, labels) in enumerate(pbar):
            labels = labels.to(DEVICE); B = len(images_pil)
            sig_optimizer.zero_grad()
            sims  = siglip_forward(siglip_model, siglip_processor, images_pil, flat_texts, B, DEVICE)
            scale = siglip_model.logit_scale.exp().clamp(max=100)
            loss  = F.cross_entropy(sims * scale, labels)
            loss.backward(); sig_optimizer.step(); total_loss += loss.item()
            pbar.set_postfix(loss=f'{total_loss/(step+1):.4f}')
            if FT_MAX_STEPS and step + 1 >= FT_MAX_STEPS:
                break
        siglip_model.eval(); val_correct = 0
        with torch.no_grad():
            for images_pil, flat_texts, labels in tqdm(sig_val_loader, desc=f'  val {epoch}', leave=False):
                labels = labels.to(DEVICE); B = len(images_pil)
                sims = siglip_forward(siglip_model, siglip_processor, images_pil, flat_texts, B, DEVICE)
                val_correct += (sims.argmax(dim=1) == labels).sum().item()
        val_acc = val_correct / len(val_samples); avg_loss = total_loss / (step + 1)
        mark = ''
        if val_acc > sig_best_val:
            sig_best_val = val_acc
            torch.save(copy.deepcopy(siglip_model.state_dict()), CKPT_SIG_PATH)
            mark = '  → saved best'
        print(f'Epoch {epoch}/{FT_EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}{mark}')
    print(f'\nBest SigLIP val_acc: {sig_best_val:.4f}')

Fine-tuning SigLIP 2: 5 epochs, lr=2e-06, bs=16



SigLIP epoch 1/5:   0%|          | 0/697 [00:00<?, ?it/s]

  val 1:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 1/5  loss=1.3182  val_acc=0.3644  → saved best


SigLIP epoch 2/5:   0%|          | 0/697 [00:00<?, ?it/s]

...

  val 4:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 4/5  loss=0.9539  val_acc=0.3991  → saved best


SigLIP epoch 5/5:   0%|          | 0/697 [00:00<?, ?it/s]

  val 5:   0%|          | 0/80 [00:00<?, ?it/s]

Epoch 5/5  loss=0.8308  val_acc=0.3904

Best SigLIP val_acc: 0.3991


In [35]:
# SigLIP 2 fine-tuning (FP32) — прогресс через обычный tqdm (без ipywidgets)
import copy, time
import torch.nn.functional as F
from tqdm import tqdm

def siglip_forward(model, processor, images_pil, flat_texts, B, device):
    images_rep = [img for img in images_pil for _ in range(N_OPTIONS)]
    inputs = processor(text=flat_texts, images=images_rep,
                       return_tensors='pt', padding='max_length', truncation=True).to(device)
    text_inputs = {k: v for k, v in inputs.items() if k != 'pixel_values'}
    img_feats = model.get_image_features(pixel_values=inputs['pixel_values'])
    txt_feats = model.get_text_features(**text_inputs)
    img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)
    txt_feats = txt_feats / txt_feats.norm(dim=-1, keepdim=True)
    img_feats = img_feats.view(B, N_OPTIONS, -1)[:, 0, :].contiguous()
    txt_feats = txt_feats.view(B, N_OPTIONS, -1).contiguous()
    return torch.bmm(img_feats.unsqueeze(1), txt_feats.transpose(1, 2)).squeeze(1)

if not RETRAIN:
    assert CKPT_SIG_PATH.exists(), f'Нет чекпоинта {CKPT_SIG_PATH}. Поставьте RETRAIN=True.'
    print(f'RETRAIN=False → обучение SigLIP пропущено, используется {CKPT_SIG_PATH.name}')
else:
    sig_optimizer = torch.optim.AdamW(siglip_model.parameters(), lr=SIG_LR, weight_decay=0.01)
    sig_best_val  = 0.0
    print(f'Fine-tuning SigLIP 2: {FT_EPOCHS} epochs, lr={SIG_LR}, bs={SIG_BATCH}\n')
    for epoch in range(1, FT_EPOCHS + 1):
        siglip_model.train()
        total_loss = 0.0
        pbar = tqdm(sig_train_loader, desc=f'SigLIP epoch {epoch}/{FT_EPOCHS}', leave=False)
        for step, (images_pil, flat_texts, labels) in enumerate(pbar):
            labels = labels.to(DEVICE); B = len(images_pil)
            sig_optimizer.zero_grad()
            sims  = siglip_forward(siglip_model, siglip_processor, images_pil, flat_texts, B, DEVICE)
            scale = siglip_model.logit_scale.exp().clamp(max=100)
            loss  = F.cross_entropy(sims * scale, labels)
            loss.backward(); sig_optimizer.step(); total_loss += loss.item()
            pbar.set_postfix(loss=f'{total_loss/(step+1):.4f}')
            if FT_MAX_STEPS and step + 1 >= FT_MAX_STEPS:
                break
        siglip_model.eval(); val_correct = 0
        with torch.no_grad():
            for images_pil, flat_texts, labels in tqdm(sig_val_loader, desc=f'  val {epoch}', leave=False):
                labels = labels.to(DEVICE); B = len(images_pil)
                sims = siglip_forward(siglip_model, siglip_processor, images_pil, flat_texts, B, DEVICE)
                val_correct += (sims.argmax(dim=1) == labels).sum().item()
        val_acc = val_correct / len(val_samples); avg_loss = total_loss / (step + 1)
        mark = ''
        if val_acc > sig_best_val:
            sig_best_val = val_acc
            torch.save(copy.deepcopy(siglip_model.state_dict()), CKPT_SIG_PATH)
            mark = '  → saved best'
        print(f'Epoch {epoch}/{FT_EPOCHS}  loss={avg_loss:.4f}  val_acc={val_acc:.4f}{mark}')
    print(f'\nBest SigLIP val_acc: {sig_best_val:.4f}')

Fine-tuning SigLIP 2: 5 epochs, lr=2e-06, bs=16



Epoch 1/5  loss=0.7285  val_acc=0.3738  → saved best


Epoch 2/5  loss=0.6584  val_acc=0.3659


Epoch 3/5  loss=0.5976  val_acc=0.3746  → saved best


Epoch 4/5  loss=0.5455  val_acc=0.3738


Epoch 5/5  loss=0.5031  val_acc=0.3715

Best SigLIP val_acc: 0.3746
